In [2]:
import pandas as pd
from pathlib import Path

In [56]:
RAW_DATA_DIR = Path.cwd().parent / "data" / "raw"
PROCESSED_DATA_DIR = Path.cwd().parent / "data" / "processed"

In [57]:
# read 10 minute interval data

ten_min_dfs = []
for file in RAW_DATA_DIR.glob("produkt_zehn*.txt"):
    df = pd.read_csv(file, sep=";", encoding="ISO-8859-1", low_memory=False, parse_dates=["MESS_DATUM"])
    ten_min_dfs.append(df)


ten_min = pd.concat(ten_min_dfs, ignore_index=True)

In [58]:
# read daily data

daily_dfs = []
for file in RAW_DATA_DIR.glob("produkt_nieder_tag_*.txt"):
    df = pd.read_csv(file, sep=";", encoding="ISO-8859-1", low_memory=False, parse_dates=["MESS_DATUM"])
    daily_dfs.append(df)

daily = pd.concat(daily_dfs, ignore_index=True)

In [59]:
ten_min["date"] = ten_min["MESS_DATUM"].dt.date

In [61]:
ten_min_agg = ten_min.groupby(["STATIONS_ID", "date"])["RWS_10"].sum().reset_index()

In [63]:
daily = daily[["STATIONS_ID", "MESS_DATUM", "  RS"]]

In [65]:
daily.rename(columns={"MESS_DATUM": "date", "  RS": "RS"}, inplace=True)
ten_min_agg.rename(columns={"RWS_10": "RS"}, inplace=True)

In [74]:
ten_min_agg["date"] = pd.to_datetime(ten_min_agg["date"])

In [75]:
full = pd.concat([daily, ten_min_agg], ignore_index=True)

In [78]:
full.drop_duplicates(subset=["STATIONS_ID", "date"], keep="last", inplace=True)

In [114]:
full["RS"] = full["RS"].where(full["RS"] >= 0, 0)

In [100]:
(full.sort_values(by=["STATIONS_ID", "date"], ascending=[True, False])
     #.groupby("STATIONS_ID")
     #.head(50)
     .to_csv(PROCESSED_DATA_DIR / "rainfall_per_day.csv", index=False, sep=";")
     )

In [103]:
full[full["date"] == "2026-07-15"]

,STATIONS_ID,date,RS
100016,2712,2026-07-15,44.92
100566,5229,2026-07-15,5.60
101116,5731,2026-07-15,10.65
101666,6258,2026-07-15,20.84
102216,6263,2026-07-15,51.26


In [110]:
(full[(full["RS"] >= 40) & (full["STATIONS_ID"].isin([2712, 6263]))]
     .sort_values(by=["STATIONS_ID", "date"], ascending=[True, False])
     #.groupby("STATIONS_ID")
     #.head(50)
     .to_csv(PROCESSED_DATA_DIR / "rainfall_per_day.csv", index=False, sep=";")
     )

In [121]:
(full[(full["date"] >= "2026-06-01") &  (full["STATIONS_ID"].isin([2712, 6263]))]
 .pivot(index="date", columns="STATIONS_ID", values="RS")
 .reset_index()
 .to_csv(PROCESSED_DATA_DIR / "rainfall_per_day_since_2026-06-01.csv", index=False, sep=";"))

In [122]:
sommer_26 = full[(full["date"] >= "2026-06-01") &  (full["STATIONS_ID"].isin([2712, 6263]))]

In [125]:
target = pd.Timestamp("2026-07-15")

sommer_26["date_group"] = sommer_26["date"].where(
    sommer_26["date"] == target,
    "2026-06-01 - 2026-07-14"
)

/tmp/ipykernel_881346/2640980981.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sommer_26["date_group"] = sommer_26["date"].where(


In [127]:
wide = (
    sommer_26.pivot_table(
        index="date_group",
        columns="STATIONS_ID",
        values="RS",
        aggfunc="sum"   # or "mean", "first", etc.
    )
    .reset_index()
)

In [129]:
wide.to_csv(PROCESSED_DATA_DIR / "15_juli_vs_rest.csv", index=False, sep=";")

In [143]:
(full[(full["date"] >= "2025-07-15") &  (full["STATIONS_ID"].isin([2712]))]
 .sort_values(by=["STATIONS_ID", "RS"], ascending=[True, False]).head(10)
 .to_csv(PROCESSED_DATA_DIR / "rainfall_last_year_KN.csv", index=False, sep=";")
)

In [144]:
(full[(full["date"] >= "2025-07-15") &  (full["STATIONS_ID"].isin([6263]))]
 .sort_values(by=["STATIONS_ID", "RS"], ascending=[True, False]).head(10)
 .to_csv(PROCESSED_DATA_DIR / "rainfall_last_year_SI.csv", index=False, sep=";")
)